In [2]:
import os
os.chdir("../")

In [1]:
%pwd

'c:\\Users\\narra\\OneDrive\\Desktop\\cibil_score_prediction\\research'

In [3]:
%pwd

'c:\\Users\\narra\\OneDrive\\Desktop\\cibil_score_prediction'

In [4]:
import pandas as pd

In [5]:
data=pd.read_csv("artifacts/data_ingestion/cibil_score_dataset_final.csv")
data.head()

,Name,Age,Occupation,Bank,Number_of_Banks,Number_of_Loans,Due_Loans,Hard_Checks,Credit_Limit,Credit_Usage,Monthly_Income,Total_Limit,Debt_to_Income_Ratio,Score_Category
0,Jane Smith,37,Lawyer,Axis Bank,2,3,0,10,438323,322302,78380,469593,4.11,Low
1,Eva White,22,Engineer,HDFC Bank,1,4,1,8,365631,78948,40911,411628,1.93,Good
2,Daniel White,34,Consultant,ICICI Bank,5,5,0,2,416026,188696,64608,434782,2.92,Standard
3,Alex Anderson,41,Doctor,ICICI Bank,1,7,1,5,494331,200883,83522,523657,2.41,Standard
4,John Harris,54,Doctor,Bank of Baroda,4,2,2,4,484846,279845,72550,498598,3.86,Standard


In [6]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Name                  1000 non-null   object 
 1   Age                   1000 non-null   int64  
 2   Occupation            1000 non-null   object 
 3   Bank                  1000 non-null   object 
 4   Number_of_Banks       1000 non-null   int64  
 5   Number_of_Loans       1000 non-null   int64  
 6   Due_Loans             1000 non-null   int64  
 7   Hard_Checks           1000 non-null   int64  
 8   Credit_Limit          1000 non-null   int64  
 9   Credit_Usage          1000 non-null   int64  
 10  Monthly_Income        1000 non-null   int64  
 11  Total_Limit           1000 non-null   int64  
 12  Debt_to_Income_Ratio  1000 non-null   float64
 13  Score_Category        1000 non-null   object 
dtypes: float64(1), int64(9), object(4)
memory usage: 109.5+ KB


In [8]:
data.columns

Index(['Name', 'Age', 'Occupation', 'Bank', 'Number_of_Banks',
       'Number_of_Loans', 'Due_Loans', 'Hard_Checks', 'Credit_Limit',
       'Credit_Usage', 'Monthly_Income', 'Total_Limit', 'Debt_to_Income_Ratio',
       'Score_Category'],
      dtype='object')

In [9]:
data.dtypes


Name                     object
Age                       int64
Occupation               object
Bank                     object
Number_of_Banks           int64
Number_of_Loans           int64
Due_Loans                 int64
Hard_Checks               int64
Credit_Limit              int64
Credit_Usage              int64
Monthly_Income            int64
Total_Limit               int64
Debt_to_Income_Ratio    float64
Score_Category           object
dtype: object

In [10]:
data.isnull().sum()

Name                    0
Age                     0
Occupation              0
Bank                    0
Number_of_Banks         0
Number_of_Loans         0
Due_Loans               0
Hard_Checks             0
Credit_Limit            0
Credit_Usage            0
Monthly_Income          0
Total_Limit             0
Debt_to_Income_Ratio    0
Score_Category          0
dtype: int64

In [11]:
from dataclasses import dataclass
from pathlib import Path

#using dataclass to avoid defining init and self  
@dataclass(frozen=True)
class DataValidationConfig:
    root_dir: Path
    STATUS_FILE: str
    unzip_data_dir: Path
    all_schema: dict

In [12]:
from mlProject.constants import *
from mlProject.utils.common import read_yaml, create_directories

In [13]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_data_validation_config(self) -> DataValidationConfig:
        config = self.config.data_validation
        schema = self.schema.COLUMNS

        create_directories([config.root_dir])

        data_validation_config = DataValidationConfig(
            root_dir=config.root_dir,
            STATUS_FILE=config.STATUS_FILE,
            unzip_data_dir = config.unzip_data_dir,
            all_schema=schema,
        )

        return data_validation_config

In [14]:
import os
from mlProject import logger

In [15]:
class DataValiadtion:
    def __init__(self, config: DataValidationConfig):
        self.config = config


    def validate_all_columns(self)-> bool:
        try:
            validation_status = None

            data = pd.read_csv(self.config.unzip_data_dir)
            all_cols = list(data.columns)

            all_schema = self.config.all_schema.keys()

            
            for col in all_cols:
                if col not in all_schema:
                    validation_status = False
                    with open(self.config.STATUS_FILE, 'w') as f:
                        f.write(f"Validation status: {validation_status}")
                else:
                    validation_status = True
                    with open(self.config.STATUS_FILE, 'w') as f:
                        f.write(f"Validation status: {validation_status}")

            return validation_status
        
        except Exception as e:
            raise e



In [18]:
try:
    config = ConfigurationManager()
    data_validation_config = config.get_data_validation_config()
    data_validation = DataValiadtion(config=data_validation_config)
    data_validation.validate_all_columns()
except Exception as e:
    raise e

[2025-09-01 18:33:48,816: INFO: common: yaml file: config\config.yaml loaded successfully]
[2025-09-01 18:33:48,819: INFO: common: yaml file: params.yaml loaded successfully]
[2025-09-01 18:33:48,835: INFO: common: yaml file: schema.yaml loaded successfully]
[2025-09-01 18:33:48,837: INFO: common: created directory at: artifacts]
[2025-09-01 18:33:48,839: INFO: common: created directory at: artifacts/data_validation]
